In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/parveenchauhan02/house-price1/socal2.csv
/kaggle/input/datasets/parveenchauhan02/house-price1/socal2/socal_pics/7981.jpg
/kaggle/input/datasets/parveenchauhan02/house-price1/socal2/socal_pics/12666.jpg
/kaggle/input/datasets/parveenchauhan02/house-price1/socal2/socal_pics/13288.jpg
/kaggle/input/datasets/parveenchauhan02/house-price1/socal2/socal_pics/6234.jpg
/kaggle/input/datasets/parveenchauhan02/house-price1/socal2/socal_pics/1269.jpg
/kaggle/input/datasets/parveenchauhan02/house-price1/socal2/socal_pics/3863.jpg
/kaggle/input/datasets/parveenchauhan02/house-price1/socal2/socal_pics/6241.jpg
/kaggle/input/datasets/parveenchauhan02/house-price1/socal2/socal_pics/10304.jpg
/kaggle/input/datasets/parveenchauhan02/house-price1/socal2/socal_pics/623.jpg
/kaggle/input/datasets/parveenchauhan02/house-price1/socal2/socal_pics/2193.jpg
/kaggle/input/datasets/parveenchauhan02/house-price1/socal2/socal_pics/14143.jpg
/kaggle/input/datasets/parveenchauhan02/house-price1/

In [2]:
import copy
import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

import matplotlib.pyplot as plt

In [3]:
CSV_PATH = "/kaggle/input/datasets/parveenchauhan02/house-price1/socal2.csv"
IMAGE_FOLDER = "/kaggle/input/datasets/parveenchauhan02/house-price1/socal2/socal_pics"

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 40
LR = 1e-4
WEIGHT_DECAY = 1e-5
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
SEED = 42
PATIENCE = 8  # early stopping patience (epochs without val R2 improvement)
BEST_MODEL_PATH = "/kaggle/working/best_model.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.manual_seed(SEED)
np.random.seed(SEED)

Using device: cuda


In [4]:
df = pd.read_csv(CSV_PATH)
print("Raw columns:", list(df.columns))
print(df.head())

Raw columns: ['image_id', 'street', 'citi', 'n_citi', 'bed', 'bath', 'sqft', 'price']
   image_id                 street             citi  n_citi  bed  bath  sqft  \
0         0  1317 Van Buren Avenue  Salton City, CA     317    3   2.0  1560   
1         1         124 C Street W      Brawley, CA      48    3   2.0   713   
2         2        2304 Clark Road     Imperial, CA     152    3   1.0   800   
3         3     755 Brawley Avenue      Brawley, CA      48    3   1.0  1082   
4         4  2207 R Carrillo Court     Calexico, CA      55    4   3.0  2547   

    price  
0  201900  
1  228500  
2  273950  
3  350000  
4  385100  


In [5]:
id_col_candidates = ["image_id", "id", "image"]
price_col_candidates = ["price", "Price"]

id_col = next((c for c in id_col_candidates if c in df.columns), None)
price_col = next((c for c in price_col_candidates if c in df.columns), None)

if id_col is None or price_col is None:
    raise ValueError(
        f"Could not find id/price columns automatically. "
        f"Columns found: {list(df.columns)}. Please set id_col/price_col manually."
    )

print(f"Using id_col='{id_col}', price_col='{price_col}'")

Using id_col='image_id', price_col='price'


In [6]:
def build_image_path(image_id):
    # dataset images are typically named like "<id>.jpg"
    return os.path.join(IMAGE_FOLDER, f"{image_id}.jpg")

df["image_path"] = df[id_col].apply(build_image_path)

# Drop rows whose image file doesn't actually exist
df["image_exists"] = df["image_path"].apply(os.path.exists)
missing = (~df["image_exists"]).sum()
if missing > 0:
    print(f"Warning: dropping {missing} rows with missing image files.")
df = df[df["image_exists"]].reset_index(drop=True)

# Drop obvious non-feature / leakage columns
drop_cols = [id_col, price_col, "image_path", "image_exists"]
categorical_candidates = [c for c in ["citi", "n_citi", "street", "citi_type"] if c in df.columns]
numeric_candidates = [
    c for c in df.columns
    if c not in drop_cols + categorical_candidates and pd.api.types.is_numeric_dtype(df[c])
]

print("Numeric tabular features:", numeric_candidates)
print("Categorical tabular features:", categorical_candidates)

Numeric tabular features: ['bed', 'bath', 'sqft']
Categorical tabular features: ['citi', 'n_citi', 'street']


In [7]:
final_cat_cols = []
for c in categorical_candidates:
    if df[c].nunique() <= 50:
        final_cat_cols.append(c)
    else:
        print(f"Dropping high-cardinality categorical column '{c}' (nunique={df[c].nunique()})")

df_encoded = pd.get_dummies(df, columns=final_cat_cols, dummy_na=False)

tabular_feature_cols = numeric_candidates + [
    c for c in df_encoded.columns
    if any(c.startswith(f"{cat}_") for cat in final_cat_cols)
]

# Drop rows with NaNs in features/target
df_encoded = df_encoded.dropna(subset=tabular_feature_cols + [price_col]).reset_index(drop=True)

print(f"Final tabular feature count: {len(tabular_feature_cols)}")

Dropping high-cardinality categorical column 'citi' (nunique=415)
Dropping high-cardinality categorical column 'n_citi' (nunique=415)
Dropping high-cardinality categorical column 'street' (nunique=12401)
Final tabular feature count: 3


In [8]:
train_df, temp_df = train_test_split(df_encoded, test_size=(VAL_SPLIT + TEST_SPLIT), random_state=SEED)
val_df, test_df = train_test_split(
    temp_df, test_size=TEST_SPLIT / (VAL_SPLIT + TEST_SPLIT), random_state=SEED
)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 10831, Val: 2321, Test: 2322


In [9]:
tab_scaler = StandardScaler()
train_df.loc[:, tabular_feature_cols] = tab_scaler.fit_transform(train_df[tabular_feature_cols])
val_df.loc[:, tabular_feature_cols] = tab_scaler.transform(val_df[tabular_feature_cols])
test_df.loc[:, tabular_feature_cols] = tab_scaler.transform(test_df[tabular_feature_cols])

# Scale target (log-transform + standardize helps regression stability for price)
train_df["log_price"] = np.log1p(train_df[price_col])
val_df["log_price"] = np.log1p(val_df[price_col])
test_df["log_price"] = np.log1p(test_df[price_col])

price_scaler = StandardScaler()
train_df["target"] = price_scaler.fit_transform(train_df[["log_price"]])
val_df["target"] = price_scaler.transform(val_df[["log_price"]])
test_df["target"] = price_scaler.transform(test_df[["log_price"]])

/tmp/ipykernel_58/2472539714.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 0.47091569  1.43345348  0.47091569 ...  0.47091569 -0.49162211
 -0.49162211]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_df.loc[:, tabular_feature_cols] = tab_scaler.fit_transform(train_df[tabular_feature_cols])
/tmp/ipykernel_58/2472539714.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 0.9964953   2.14360837 -0.00538873 ... -0.61279933 -0.62849977
  0.3979162 ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_df.loc[:, tabular_feature_cols] = tab_scaler.fit_transform(train_df[tabular_feature_cols])
/tmp/ipykernel_58/2472539714.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 0.47091

In [10]:
print(train_df["log_price"])

10186    13.384575
598      14.456839
4562     13.710695
9953     13.384729
14347    13.737550
           ...    
5191     14.237506
13418    13.996999
5390     12.429220
860      12.560248
7270     13.384575
Name: log_price, Length: 10831, dtype: float64


In [11]:
print(val_df["log_price"])

10740    14.148406
1450     12.996802
1097     12.725464
2094     13.335862
9271     12.947774
           ...    
6021     12.765406
2788     13.591117
10003    13.384575
180      12.703816
46       12.296832
Name: log_price, Length: 2321, dtype: float64


In [12]:
print(test_df["log_price"])

8140     13.840204
10871    14.151984
578      14.315680
6434     13.100120
14831    12.923915
           ...    
3578     12.733758
5714     12.765691
73       12.254391
3611     12.899197
8879     12.733758
Name: log_price, Length: 2322, dtype: float64


In [13]:
print(test_df["target"])

8140     0.995095
10871    1.606542
578      1.927575
6434    -0.456322
14831   -0.801886
           ...   
3578    -1.174812
5714    -1.112186
73      -2.114923
3611    -0.850361
8879    -1.174812
Name: target, Length: 2322, dtype: float64


In [14]:
print(val_df["target"])

10740    1.599525
1450    -0.658943
1097    -1.191077
2094     0.006005
9271    -0.755094
           ...   
6021    -1.112747
2788     0.506599
10003    0.101539
180     -1.233533
46      -2.031691
Name: target, Length: 2321, dtype: float64


In [15]:
print(train_df["target"])

10186    0.101539
598      2.204408
4562     0.741109
9953     0.101840
14347    0.793775
           ...   
5191     1.774263
13418    1.302594
5390    -1.772057
860     -1.515092
7270     0.101539
Name: target, Length: 10831, dtype: float64


In [16]:
def inverse_target(scaled_target):
    """Undo standardization + log1p to get back to raw price."""
    log_price = price_scaler.inverse_transform(scaled_target.reshape(-1, 1)).flatten()
    return np.expm1(log_price)


In [17]:
train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [18]:
print(train_transform)

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.9, 1.1), contrast=(0.9, 1.1), saturation=None, hue=None)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [19]:
class HouseDataset(Dataset):
    def __init__(self, dataframe, tabular_cols, id_col, transform):
        self.df = dataframe.reset_index(drop=True)
        self.tabular_cols = tabular_cols
        self.id_col = id_col
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = build_image_path(row[self.id_col])
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        tabular = torch.tensor(row[self.tabular_cols].values.astype(np.float32))
        target = torch.tensor(row["target"], dtype=torch.float32)
        return image, tabular, target


In [20]:
train_ds = HouseDataset(train_df, tabular_feature_cols, id_col, train_transform)
print(train_ds)

In [21]:
train_ds = HouseDataset(train_df, tabular_feature_cols, id_col, train_transform)
val_ds = HouseDataset(val_df, tabular_feature_cols, id_col, eval_transform)
test_ds = HouseDataset(test_df, tabular_feature_cols, id_col, eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [22]:
class CNNBranch(nn.Module):
    """Pretrained ResNet18 backbone, output = image feature vector."""
    def __init__(self, out_dim=256, freeze_backbone=False):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()  # remove classification head
        self.backbone = backbone

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.project = nn.Sequential(
            nn.Linear(in_features, out_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

    def forward(self, x):
        feats = self.backbone(x)
        return self.project(feats)

In [23]:
class TabularBranch(nn.Module):
    """Simple MLP branch for structured features."""
    def __init__(self, in_dim, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(128, out_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


In [24]:
class MultimodalHousePriceModel(nn.Module):
    def __init__(self, tabular_in_dim, img_feat_dim=256, tab_feat_dim=64, freeze_backbone=False):
        super().__init__()
        self.cnn_branch = CNNBranch(out_dim=img_feat_dim, freeze_backbone=freeze_backbone)
        self.tabular_branch = TabularBranch(tabular_in_dim, out_dim=tab_feat_dim)

        concat_dim = img_feat_dim + tab_feat_dim
        self.regression_head = nn.Sequential(
            nn.Linear(concat_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, image, tabular):
        img_feats = self.cnn_branch(image)
        tab_feats = self.tabular_branch(tabular)
        combined = torch.cat([img_feats, tab_feats], dim=1)
        out = self.regression_head(combined)
        return out.squeeze(1)

In [25]:
model = MultimodalHousePriceModel(
    tabular_in_dim=len(tabular_feature_cols),
    img_feat_dim=256,
    tab_feat_dim=64,
    freeze_backbone=False,  # set True first to warm up the head, then unfreeze for fine-tuning
).to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 180MB/s] 


In [26]:
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

In [27]:
def run_epoch(loader, model, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_preds, all_targets = [], []

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for images, tabular, targets in loader:
            images, tabular, targets = images.to(device), tabular.to(device), targets.to(device)

            if is_train:
                optimizer.zero_grad()

            preds = model(images, tabular)
            loss = criterion(preds, targets)

            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            all_preds.append(preds.detach().cpu().numpy())
            all_targets.append(targets.detach().cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    avg_loss = total_loss / len(loader.dataset)
    r2 = r2_score(all_targets, all_preds)
    return avg_loss, r2, all_preds, all_targets

In [28]:
best_val_r2 = -np.inf
best_state = None
epochs_no_improve = 0

In [29]:
history = {"train_loss": [], "val_loss": [], "train_r2": [], "val_r2": []}


In [30]:
for epoch in range(1, EPOCHS + 1):
    train_loss, train_r2, _, _ = run_epoch(train_loader, model, criterion, optimizer)
    val_loss, val_r2, _, _ = run_epoch(val_loader, model, criterion, optimizer=None)

    scheduler.step(val_r2)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_r2"].append(train_r2)
    history["val_r2"].append(val_r2)

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} R2: {train_r2:.4f} | "
          f"Val Loss: {val_loss:.4f} R2: {val_r2:.4f}")

    if val_r2 > best_val_r2:
        best_val_r2 = val_r2
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, BEST_MODEL_PATH)
        epochs_no_improve = 0
        print(f"  -> New best model saved (Val R2: {best_val_r2:.4f})")
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping triggered after {epoch} epochs (no improvement in {PATIENCE} epochs).")
        break


Epoch 01/40 | Train Loss: 0.6736 R2: 0.3264 | Val Loss: 0.5058 R2: 0.4839
  -> New best model saved (Val R2: 0.4839)
Epoch 02/40 | Train Loss: 0.4798 R2: 0.5202 | Val Loss: 0.4787 R2: 0.5115
  -> New best model saved (Val R2: 0.5115)
Epoch 03/40 | Train Loss: 0.3730 R2: 0.6270 | Val Loss: 0.4925 R2: 0.4974
Epoch 04/40 | Train Loss: 0.2801 R2: 0.7199 | Val Loss: 0.4687 R2: 0.5217
  -> New best model saved (Val R2: 0.5217)
Epoch 05/40 | Train Loss: 0.2094 R2: 0.7906 | Val Loss: 0.4455 R2: 0.5453
  -> New best model saved (Val R2: 0.5453)
Epoch 06/40 | Train Loss: 0.1732 R2: 0.8268 | Val Loss: 0.4478 R2: 0.5431
Epoch 07/40 | Train Loss: 0.1477 R2: 0.8523 | Val Loss: 0.4188 R2: 0.5727
  -> New best model saved (Val R2: 0.5727)
Epoch 08/40 | Train Loss: 0.1327 R2: 0.8673 | Val Loss: 0.4221 R2: 0.5693
Epoch 09/40 | Train Loss: 0.1177 R2: 0.8823 | Val Loss: 0.4269 R2: 0.5643
Epoch 10/40 | Train Loss: 0.1046 R2: 0.8954 | Val Loss: 0.4072 R2: 0.5845
  -> New best model saved (Val R2: 0.5845)
Ep

In [31]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
test_loss, test_r2, test_preds, test_targets = run_epoch(test_loader, model, criterion, optimizer=None)

test_preds_raw = inverse_target(test_preds)
test_targets_raw = inverse_target(test_targets)
mae_raw = mean_absolute_error(test_targets_raw, test_preds_raw)

print("\n===== FINAL TEST RESULTS (best model) =====")
print(f"Test R2 (scaled target): {test_r2:.4f}")
print(f"Test MAE (raw price $):  {mae_raw:,.0f}")


===== FINAL TEST RESULTS (best model) =====
Test R2 (scaled target): 0.5854
Test MAE (raw price $):  166,961


In [34]:
print(tabular_feature_cols)

['bed', 'bath', 'sqft']


In [35]:
print(eval_transform)

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
